# Free open-model baseline: qwen2.5:7b on Colab

Runs the same five LLM conditions as the Claude runs (same prompts, same code) with the open model `qwen2.5:7b` via Ollama, on Colab's free GPU. Nothing is installed on your own computer.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then run the cells top to bottom (Shift+Enter).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install Ollama and the model (on Colab's machine)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5:7b
!ollama list

## 2. Get the code (public GitHub branch) and upload the inputs

When the upload button appears, choose **`colab_inputs.zip`** from `E:\Documents\MCP For Job Apply\`.

In [ ]:
!git clone -q -b v1b-gold-labelling https://github.com/IhsanKabir/policy-extraction-rag.git
%cd policy-extraction-rag
!git log --oneline -1
!pip -q install pydantic
from google.colab import files
up = files.upload()
!unzip -o -q colab_inputs.zip -d data/ && ls -la data/

## 3. Quick check: 3 records

In [ ]:
!python -m extractors.llm_pipeline --model qwen2.5:7b --mode llm --clauses data/v1c_target_clauses.jsonl --output data/v1b_llm_qwen7b.jsonl --max-records 3

## 4. The five full runs (181 records each; resumable: if Colab disconnects, re-run cells 1, 2 and this one)

Plain, citations, citations + context, and the two guideline conditions (same prompts as the Claude runs).

In [ ]:
C = '--model qwen2.5:7b --clauses data/v1c_target_clauses.jsonl'
G = '--guidelines docs/prompt_guidelines.txt'
!python -m extractors.llm_pipeline $C --mode llm --output data/v1b_llm_qwen7b.jsonl
!python -m extractors.llm_pipeline $C --mode llm_cited --output data/v1b_llm_cited_qwen7b.jsonl
!python -m extractors.llm_pipeline $C --mode llm_cited --with-chunk-context --chunks data/chunks.jsonl --output data/v1b_llm_cited_ctx_qwen7b.jsonl
!python -m extractors.llm_pipeline $C --mode llm $G --output data/v1b_llm_guided_qwen7b.jsonl
!python -m extractors.llm_pipeline $C --mode llm_cited $G --output data/v1b_llm_cited_guided_qwen7b.jsonl

## 5. Download the results

Save `qwen7b_results.zip` into `E:\Documents\MCP For Job Apply\` and tell Claude it's there.

In [ ]:
!ollama show qwen2.5:7b --modelfile | head -3 > data/qwen_model_info.txt
!cd data && zip -q ../qwen7b_results.zip v1b_*qwen7b.jsonl qwen_model_info.txt
from google.colab import files
files.download('qwen7b_results.zip')